### Building Agents
* Part 1: A simple "Agent" and "Agent Loop". Basically an LLM call. We'll add tracing and streaming to the mix.
* Part 2: Adding a Tool.
* Part 3: Adding Memory

In [ ]:
# The imports

import os
import requests
from dotenv import load_dotenv
from openai.types.responses import ResponseTextDeltaEvent
from agents import Agent, Runner, trace, function_tool, SQLiteSession
import agents
load_dotenv(override=True)


### What is an SDK?

An SDK (Software Development Kit) is a collection of tools, libraries, APIs, and documentation that helps developers build applications for a particular platform or service.

These classes and functions are provided by the OpenAI Agents SDK, whose PyPI package is typically named openai-agents.

In [ ]:
print(Agent.__module__)
print(agents.__file__)

In [ ]:
from importlib.metadata import packages_distributions

print(packages_distributions()["agents"])

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv("/Users/nareshchaurasia/nc/PYTHON-ARCHITECT/Python-Immersive-AI-MAC/.env")

api_key = os.getenv("CO_API_KEY")
print(api_key)

In [ ]:
# This __replaces__ the default OpenAI-backend processor with LangSmith's, so traces get sent to your LangSmith project instead of `platform.openai.com/traces`.

from langsmith.wrappers import OpenAIAgentsTracingProcessor
from agents import set_trace_processors

set_trace_processors([OpenAIAgentsTracingProcessor()])

In [ ]:
from langsmith import Client
client = Client()
print(client.list_projects(limit=1))  # should not raise 401 if key is valid

In [ ]:

# Make an agent with name, instructions, model

agent = Agent(name="Jokester", instructions="You are a joke teller", model="gpt-5.4-mini")

In [ ]:
# Run the joke with Runner.run(agent, prompt)
# LangSmith - Agent workflow
result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")


In [ ]:
# Here is the final output

print(result.final_output)

In [ ]:
# Here is the detail of the LLM calls

result.to_input_list()

### Adding Observability with a trace

**Where traces are stored**

By default, traces from the OpenAI Agents SDK (`agents` package, used with `Runner.run`) are sent to __OpenAI's servers__ and viewable in the __OpenAI Platform dashboard__, specifically at:

__[](https://platform.openai.com/traces)<https://platform.openai.com/traces>__ (or similarly named "Traces"/"Logs" section under your OpenAI account)


In [13]:
# LangSmith
with trace("Understanding the AI Agents"):
    result = await Runner.run(agent, "Explain the concept of AI Agents")
print(result.final_output)

AI agents are software systems that can **perceive**, **decide**, and **act** toward a goal, often with some level of autonomy.

### Simple idea
Think of an AI agent like a digital assistant that doesn’t just answer questions — it can also:
- **observe** information from its environment,
- **choose** what to do next,
- **take actions** using tools or APIs,
- **repeat** this loop until it reaches a goal.

### Basic loop
An AI agent usually follows a cycle like:

1. **Input / perception**  
   It gets data from text, sensors, files, web pages, databases, etc.

2. **Reasoning / planning**  
   It figures out what the goal is and what steps might achieve it.

3. **Action**  
   It performs an action, like sending a message, querying a database, running code, or booking something.

4. **Feedback**  
   It checks the result and adjusts its next step.

### Example
If you ask an AI agent:  
“Find the best flight for next Friday and book it if it’s under $300,”

it may:
- search flight options,

### Steaming

In [14]:
# Streaming

result = Runner.run_streamed(agent, input="Please tell me 3 jokes about AI Agents.")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

Absolutely — here are 3 AI agent jokes:

1. **Why did the AI agent bring a ladder to work?**  
   Because it wanted to reach the *next level* of autonomy.

2. **What do you call an AI agent that keeps interrupting?**  
   A *prompt* optimizer.

3. **Why was the AI agent bad at keeping secrets?**  
   It always tried to *reason things out loud*.

If you want, I can also do **darker**, **nerdier**, or **more absurd** AI agent jokes.

## Part 2: Adding a tool

In [ ]:
# pushover_user = os.getenv("PUSHOVER_USER")
# pushover_token = os.getenv("PUSHOVER_TOKEN")
# pushover_url = "https://api.pushover.net/1/messages.json"

# if pushover_user:
#     if pushover_user.startswith("u"):
#         print("Pushover user found and looks good")
#     else:
#         print("Pushover user found but doesn't start with u")
# else:
#     print("Pushover user not found")

# if pushover_token:
#     if pushover_token.startswith("a"):
#         print("Pushover token found and looks good")
#     else:
#         print("Pushover token found but doesn't start with a")
# else:
#     print("Pushover token not found")

In [15]:
# Remember this?

def push(message):
    print(f"Push: {message}")
    # payload = {"user": pushover_user, "token": pushover_token, "message": message}
    # requests.post(pushover_url, data=payload)

In [16]:
push("HEY!!")

Push: HEY!!


In [17]:
push

<function __main__.push(message)>

In [18]:
# Now this:

@function_tool
def push_tool(message: str) -> str:
    # """ Send the given message to the user as a push notification """
    # payload = {"user": pushover_user, "token": pushover_token, "message": message}
    # result = requests.post(pushover_url, data=payload).status_code
    # return f"Push sent with API status code {result}"

    """ Send the given message to the user as a push notification """
    return f"Push: {message}"

In [19]:
push_tool

FunctionTool(name='push_tool', description='Send the given message to the user as a push notification', params_json_schema={'properties': {'message': {'title': 'Message', 'type': 'string'}}, 'required': ['message'], 'title': 'push_tool_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x1116cb390>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False)

In [20]:
push_tool.description

'Send the given message to the user as a push notification'

In [21]:

notifier = Agent(name="Notifier", model="gpt-5.4-mini", instructions="You notify the user upon request", tools=[push_tool])

In [22]:
with trace("Pizza has arrived"):
    result = await Runner.run(notifier, "Notify the user that the pizza is here")

print(result.final_output)


Done.


## Now go and look at the trace

https://platform.openai.com/traces

## Part 3: Sessions (memory)

Within a Runner.run() application level turn, the conversation history is maintained.

But each call to Runner.run() is a fresh start.

Let's see that:

In [23]:
agent = Agent(name="Assistant", model="gpt-5.4-mini")

In [24]:
response = await Runner.run(agent, "Hi there. My name is Naresh")
print(response.final_output)

Hi Naresh — nice to meet you. How can I help today?


In [25]:
response = await Runner.run(agent, "What's my name?")
print(response.final_output)

I don’t know your name unless you tell me. If you want, you can share it and I’ll use it.


### Memory approach 1 - just manually pass in the list of dicts

In [26]:
response = await Runner.run(agent, "Hi there. My name is Naresh.")
print(response.final_output)

Hi Naresh — nice to meet you! How can I help you today?


In [27]:
response.to_input_list()

[{'content': 'Hi there. My name is Naresh.', 'role': 'user'},
 {'id': 'msg_08ba4c46cba74fce006ab10d1e9b5c87d2b064116eac882883',
  'content': [{'annotations': [],
    'text': 'Hi Naresh — nice to meet you! How can I help you today?',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'phase': 'final_answer'}]

In [ ]:
next_input = response.to_input_list() + [{"role": "user", "content": "What's my name?"}]
next_input

In [ ]:
response = await Runner.run(agent, next_input)
print(response.final_output)

### Another approach - use OpenAI Agents SDK built in SQLLite session

In [ ]:
# This is created in-memory
# For an on-disk memory, use SQLiteSession("12345", "memory.db")

session = SQLiteSession("id-cts-1000")

In [ ]:
response = await Runner.run(agent, "Hi there. My name is NC", session=session)
print(response.final_output)

In [ ]:
response = await Runner.run(agent, "What's my name?", session=session)
print(response.final_output)